## <font color="royalblue">**Pre_Procesamiento de datos**</font>
### <font color="royalblue">**Preparacion de datos**</font>
Primera mirada a los dataframe generados a traves de los diferentes portales.  
    - Adzuna: df_Adzuna  
    - Indeed: df_Indeed  
    - LinkedIn: df_LinkedIn  
Aplicar primeras ediciones a los datos e intentar complementar algunos de los campos por medio de busquedas sencillas dentro del cada dataframe  

In [95]:
import pandas as pd
import numpy as np
import re


In [96]:
# Dataframe de Adzuna original
df_Adzuna_orig=pd.read_csv('df_Adzuna.csv')
# Dataframe de Adzuna original con la descripcion completa obtenida a partir de la url
df_Adzuna=pd.read_csv('df_Adzuna_descripcion_editada.csv')


In [ ]:
# Agregar columna portal_web con el nombre del portal
df_Adzuna["portal_web"] = "Adzuna"
# Eliminar la columna descripcion original y renombrar la columna descripcion_final a descripcion
df_Adzuna.drop(columns=["descripcion"], inplace=True)
df_Adzuna.rename(columns={"descripcion_final": "descripcion"}, inplace=True)
 

In [98]:
df_Indeed=pd.read_csv('df_Indeed.csv')
df_Indeed["portal_web"] = "Indeed"
df_Indeed.drop(columns=['Unnamed: 0'], inplace=True) 


In [99]:
df_LinkedIn=pd.read_csv('LinkedIn.csv') 
df_LinkedIn["portal_web"] = "LinkedIn"
df_LinkedIn.drop(columns=["fecha_publicacion"], inplace=True)


### <font color="royalblue">Extraccion de solicitudes de la oferta</font>

In [ ]:
# Función para detectar modalidad de trabajo a partir de la descripcion de la oferta laboral, con un enfoque más amplio y flexible, 
# diseñada para extraer modalidad dentro de la descripcion en LinkedIn

# Definir un diccionario con las modalidades y sus posibles palabras clave
TIPO_MODALIDAD = {
    "hibrido": ["trabajo híbrido", "modalidad híbrida", "hybrid", "hybrid working", "hybrid model"],
    "remoto": ["teletrabajo", "trabajo remoto", "remoto", "remote", "remote work", "work from home", "fully remote", "remote-first"],
    "presencial": ["on-site", "presencial"]
}

def buscar_modalidad(texto):
    """Funcion que permite identificar la modalidad de trabajo (híbrido, remoto o presencial) a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()

    for modalidad, tipos in TIPO_MODALIDAD.items():
        for palabra in tipos:
            if palabra in texto:
                return modalidad

    return None


In [102]:
# Función anterior con un enfoque más directo, pero menos flexible, 
# diseñada para extraer modalidad dentro de ubicacion en Indeed

def detectar_modalidad(texto):
    """Funcion que permite identificar la modalidad de trabajo (híbrido, remoto o presencial) a partir del texto suministrado"""
    if "Trabajo híbrido" in texto:
        return "híbrido"
    if "Teletrabajo" in texto:
        return "remoto"
    return None


In [103]:
# Prueba para Deteccion de modalidades
mod_keywords = df_Adzuna["descripcion"].str.extractall(
    r"(hybrid|remote|on-site|work from home|flexible|teletrabajo|remoto|híbrido)")

mod_keywords[0].value_counts()


0
remote         17
flexible       14
hybrid         10
híbrido         2
on-site         1
teletrabajo     1
remoto          1
Name: count, dtype: int64

In [104]:
# Definir un diccionario con los tipos de contrato y sus posibles palabras clave
TIPO_CONTRATO = {
    "indefinido": ["indefinido", "permanent"],
    "temporal": ["temporal", "temporary"],
    "tiempo completo": ["tiempo completo", "full-time", "full time", "jornada completa"],
    "tiempo parcial": ["tiempo parcial", "part-time", "part time", "jornada parcial"],
    "prácticas": ["prácticas", "internship", "becario", "trainee"],
    "formación": ["formación", "apprenticeship"],
    "freelance": ["freelance", "autónomo", "self-employed", "contractor"],
    "contrato": ["contract", "contrato"],  # genérico
}

def detectar_tipo_contrato(texto):
    """"Funcion que permite identificar el tipo de contrato a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()

    for contrato, palabras in TIPO_CONTRATO.items():
        for palabra in palabras:
            if palabra in texto:
                return contrato

    return np.nan


In [105]:
# Definir un diccionario con los idiomas y sus posibles palabras clave
IDIOMAS = {
    "español": ["español", "spanish", "castellano"],
    "inglés": ["inglés", "english"],
    "catalán": ["catalán", "catalan", "català"],
    "francés": ["francés", "french"],
    "alemán": ["alemán", "german"],
    "italiano": ["italiano", "italian"],
    "portugués": ["portugués", "portuguese"],
}

def detectar_idiomas(texto):
    """Funcion que permite identificar los idiomas requeridos a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()
    encontrados = []

    for idioma, palabras in IDIOMAS.items():
        for palabra in palabras:
            if palabra in texto:
                encontrados.append(idioma)
                break

    return ", ".join(encontrados) if encontrados else np.nan


In [ ]:
# Función que permite identificar los años de experiencia requeridos a partir de la descripcion de una oferta laboral
def detectar_experiencia_años(texto):
    """Funcion que permite identificar los años de experiencia requeridos a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()

    if "sin experiencia" in texto or "no experience" in texto:
        return 0

    coincidencias = re.findall(r"(\d+)\s*\+?\s*(años|year|years)", texto)

    if not coincidencias:
        return np.nan

    numeros = [int(num) for num, _ in coincidencias]
    return max(numeros)


In [ ]:
# Definir un diccionario con los niveles de experiencia y sus posibles palabras clave
NIVELES = {
    "becario": ["becario", "prácticas", "intern", "trainee"],
    "junior": ["junior", "jr"],
    "middle": ["middle", "mid", "semi senior", "ssr"],
    "senior": ["senior", "sr", "lead", "principal"],
}

def detectar_experiencia_nivel(texto):
    """Funcion que permite identificar el nivel de experiencia requerido a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()

    for nivel, palabras in NIVELES.items():
        for palabra in palabras:
            if palabra in texto:
                return nivel

    return np.nan


In [108]:
# Definir un diccionario con los niveles educativos y sus posibles palabras clave
NIVEL_EDUCATIVO = {
    "doctorado": ["doctorado", "phd", "doctoral"],
    "master": ["máster", "master", "msc", "postgrado", "posgrado"],
    "grado": ["grado", "licenciatura", "bachelor", "bsc", "degree"],
    "fp": ["fp", "formación profesional", "ciclo formativo", "vocational"],
    "bootcamp": ["bootcamp", "curso intensivo", "certificación"]
}

def detectar_nivel_educativo(texto):
    """Funcion que permite identificar el nivel educativo requerido a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()
    encontrados = []

    for nivel, palabras in NIVEL_EDUCATIVO.items():
        for palabra in palabras:
            if palabra in texto:
                encontrados.append(nivel)
                break

    if not encontrados:
        return np.nan

    # Prioridad académica
    prioridad = ["doctorado", "master", "grado", "fp", "bootcamp"]

    for nivel in prioridad:
        if nivel in encontrados:
            return nivel

    return np.nan


In [109]:
# Definir un diccionario con las hard skills y sus posibles palabras clave
HARD_SKILLS = {
    "python": ["python"],
    "r": [" r ", " r,", " r."],
    "sql": ["sql"],
    "excel": ["excel", "microsoft excel"],
    "power bi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "aws": ["aws", "amazon web services"],
    "azure": ["azure"],
    "gcp": ["gcp", "google cloud"],
    "spark": ["spark"],
    "hadoop": ["hadoop"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "scikit-learn": ["scikit-learn", "sklearn"],
    "docker": ["docker"],
    "kubernetes": ["kubernetes", "k8s"],
    "airflow": ["airflow"],
    "git": ["git"],
    "postgresql": ["postgresql", "postgres"],
    "mysql": ["mysql"],
    "mongodb": ["mongodb", "mongo"],
}

def detectar_hard_skills(texto):
    """Funcion que permite identificar las hard skills requeridas a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()
    encontrados = []

    for skill, palabras in HARD_SKILLS.items():
        for palabra in palabras:
            if palabra in texto:
                encontrados.append(skill)
                break  # evita duplicados dentro de la misma skill

    return ", ".join(encontrados) if encontrados else np.nan


In [110]:
# Definir un diccionario con las soft skills y sus posibles palabras clave
SOFT_SKILLS = {
    "comunicación": ["comunicación", "comunicarse", "communication"],
    "trabajo en equipo": ["trabajo en equipo", "teamwork", "colaboración", "colaborative"],
    "liderazgo": ["liderazgo", "leadership", "liderar"],
    "pensamiento crítico": ["pensamiento crítico", "critical thinking"],
    "resolución de problemas": ["resolución de problemas", "problem solving"],
    "organización": ["organización", "organizado", "organizational"],
    "proactividad": ["proactividad", "proactivo", "proactive"],
    "adaptabilidad": ["adaptabilidad", "adaptable", "adaptability"],
    "gestión del tiempo": ["gestión del tiempo", "time management"],
    "orientación a resultados": ["orientación a resultados", "results oriented"],
}

def detectar_soft_skills(texto):
    """Funcion que permite identificar las soft skills requeridas a partir de la descripcion de una oferta laboral"""
    texto = texto.lower()
    encontrados = []

    for skill, palabras in SOFT_SKILLS.items():
        for palabra in palabras:
            if palabra in texto:
                encontrados.append(skill)
                break

    return ", ".join(encontrados) if encontrados else np.nan


ADZUNA

In [111]:
df_Adzuna["modalidad"] = df_Adzuna["descripcion"].apply(buscar_modalidad)
df_Adzuna["tipo_contrato"] = df_Adzuna["descripcion"].apply(detectar_tipo_contrato)
df_Adzuna["idiomas"] = df_Adzuna["descripcion"].apply(detectar_idiomas)
df_Adzuna["experiencia_años"] = df_Adzuna["descripcion"].apply(detectar_experiencia_años)
df_Adzuna["experiencia_nivel"] = df_Adzuna["descripcion"].apply(detectar_experiencia_nivel)
df_Adzuna["nivel_educativo"] = df_Adzuna["descripcion"].apply(detectar_nivel_educativo)
df_Adzuna["hard_skills"] = df_Adzuna["descripcion"].apply(detectar_hard_skills)
df_Adzuna["soft_skills"] = df_Adzuna["descripcion"].apply(detectar_soft_skills)
df_Adzuna

,titulo,empresa,ubicacion,contract_type,contract_time,category,url,id,fecha,descripcion,es_truncada,portal_web,modalidad,tipo_contrato,idiomas,experiencia_años,experiencia_nivel,nivel_educativo,hard_skills,soft_skills
0,Data Analyst,Perk,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5655420542?utm_m...,5655420542,2026-03-05T22:17:29Z,About Us Perk (formerly TravelPerk) is the int...,False,Adzuna,None,NaN,inglés,2.0,becario,master,"python, sql, excel, git",comunicación
1,Data Analyst,Veepee,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5590818342?utm_m...,5590818342,2026-01-18T22:18:49Z,Pioneer of online flash sales since 2001 and k...,False,Adzuna,None,NaN,inglés,4.0,becario,master,"python, sql",proactividad
2,Data Analyst,Propelling Tech,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5584544401?utm_m...,5584544401,2026-01-14T20:09:21Z,"As a Data Analyst , you will be part of a high...",False,Adzuna,hibrido,NaN,"español, inglés",NaN,becario,master,NaN,"comunicación, liderazgo"
3,Data Analyst,PrimeIT,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5304781614?utm_m...,5304781614,2025-07-16T00:10:07Z,Qué buscamos? A partir de 3 años de experienc...,False,Adzuna,None,formación,inglés,3.0,becario,NaN,"python, sql, power bi, tableau",NaN
4,Data Analyst Jr,Solicitud de empleo para Data Analyst Jr en DD...,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5556867176?utm_m...,5556867176,2025-12-26T20:51:02Z,"Perfil Si te apasionan los datos, quieres apre...",False,Adzuna,None,prácticas,NaN,NaN,becario,NaN,"excel, git","comunicación, trabajo en equipo"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,Senior Finance Systems Project Manager - HQ,Glovo,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5648038417?utm_m...,5648038417,2026-02-28T01:45:31Z,Senior Finance Systems Project Manager - HQ Ub...,False,Adzuna,None,NaN,NaN,NaN,senior,NaN,NaN,"comunicación, organización"
72,HQ BARCELONA - Senior Python Developer (Data T...,Jobandtalent,Barcelona,NaN,NaN,Trabajos en informática,https://www.adzuna.es/details/4800600477?utm_m...,4800600477,2024-07-30T15:53:31Z,Who we are: We are a workforce on-demand compa...,False,Adzuna,remoto,contrato,"inglés, alemán",4.0,becario,grado,"python, sql, excel, spark, docker, kubernetes,...","comunicación, liderazgo"
73,Analista de Producto,Product Madness,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5548465130?utm_m...,5548465130,2025-12-19T14:04:21Z,Estamos ampliando nuestros equipos y esta es u...,False,Adzuna,remoto,tiempo completo,inglés,NaN,becario,grado,"python, r, sql, excel, power bi, tableau",comunicación
74,Senior Data Architect,Bunge,Barcelona,NaN,NaN,Unknown,https://www.adzuna.es/details/5314694773?utm_m...,5314694773,2025-07-21T17:19:07Z,Location : Barcelona Hub City : Barcelona Stat...,False,Adzuna,hibrido,NaN,NaN,5.0,middle,master,"python, sql, excel, git","comunicación, trabajo en equipo"


INDEED

In [112]:

df_Indeed["ubicacion"] = df_Indeed["ubicacion"].astype(str)
df_Indeed["modalidad"] = df_Indeed["ubicacion"].apply(detectar_modalidad)
df_Indeed["ubicacion"] = df_Indeed["ubicacion"].apply(lambda x: x.replace("Trabajo híbrido in ", "").replace("Teletrabajo in ", "").strip())
df_Indeed["descripcion"] = df_Indeed["descripcion"].astype(str)
df_Indeed["idiomas"] = df_Indeed["descripcion"].apply(detectar_idiomas)
df_Indeed["experiencia_años"] = df_Indeed["descripcion"].apply(detectar_experiencia_años)
df_Indeed["experiencia_nivel"] =  df_Indeed["descripcion"].apply(detectar_experiencia_nivel)
df_Indeed["nivel_educativo"] = df_Indeed["descripcion"].apply(detectar_nivel_educativo) 
df_Indeed["hard_skills"] = df_Indeed["descripcion"].apply(detectar_hard_skills)
df_Indeed["soft_skills"] = df_Indeed["descripcion"].apply(detectar_soft_skills)
df_Indeed   


,id,titulo,empresa,ubicacion,salario,tipo_contrato,modalidad,url,descripcion,portal_web,idiomas,experiencia_años,experiencia_nivel,nivel_educativo,hard_skills,soft_skills
0,ef8da29b4067d8ce,FP&A Analyst,Freightos,"08018 Barcelona, Barcelona provincia",NaN,Jornada completa,híbrido,https://es.indeed.com/viewjob?jk=ef8da29b4067d8ce,About Us\nAlmost every single thing that you e...,Indeed,"español, inglés",3.0,senior,grado,"excel, power bi, tableau, git","comunicación, liderazgo, organización"
1,d01ca385b03a8ab5,BI Analyst,Grupo Planeta,"Barcelona, Barcelona provincia",NaN,NaN,None,https://es.indeed.com/viewjob?jk=d01ca385b03a8ab5,Activa | Planeta Innovación Somos el proveedor...,Indeed,NaN,40.0,NaN,grado,"python, r, sql, excel, power bi, azure, postgr...","comunicación, pensamiento crítico, gestión del..."
2,cdef0123456789ab,NaN,NaN,nan,NaN,NaN,None,https://es.indeed.com/viewjob?jk=cdef0123456789ab,nan,Indeed,NaN,NaN,NaN,NaN,NaN,NaN
3,8c331ce1dbb312a4,Junior Data Analyst,Holded,"08039 Barcelona, Barcelona provincia",NaN,Jornada completa,híbrido,https://es.indeed.com/viewjob?jk=8c331ce1dbb312a4,"Join the team. Make an impact.\nAt\nHolded\n, ...",Indeed,"español, inglés",3.0,junior,master,"python, sql, excel",NaN
4,e84341c140a96071,Junior Data Analyst,EXOGROUP,"08005 Barcelona, Barcelona provincia",NaN,Jornada completa,híbrido,https://es.indeed.com/viewjob?jk=e84341c140a96071,About ExoClick:\nExoClick is an innovative and...,Indeed,"español, inglés, portugués",NaN,becario,grado,"python, sql, git","comunicación, liderazgo"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,06e2a78e8dc73bd0,OutSystems Developer Consultant,Zurich Insurance,"Barcelona, Barcelona provincia",NaN,Jornada completa,None,https://es.indeed.com/viewjob?jk=06e2a78e8dc73bd0,We Are Waiting for You\n\nHi there!\nI am Álva...,Indeed,"español, inglés, alemán, portugués",3.0,becario,NaN,"sql, excel",NaN
135,7e0cce10992dd6d1,Consultor Inmobiliario Industrial (Barcelona),Engel & Völkers España,"Barcelona, Barcelona provincia",NaN,Jornada completa,None,https://es.indeed.com/viewjob?jk=7e0cce10992dd6d1,Descripción:\nEngel & Völkers es una empresa l...,Indeed,"español, inglés",3.0,becario,NaN,excel,"comunicación, trabajo en equipo"
136,508aa6b0a7ffa408,Senior II Back-End Engineer,Preply,"Barcelona, Barcelona provincia",NaN,Jornada completa,híbrido,https://es.indeed.com/viewjob?jk=508aa6b0a7ffa408,"We power people’s progress.\nAt Preply, we’re ...",Indeed,inglés,NaN,junior,NaN,"python, excel, aws, gcp, spark","comunicación, proactividad"
137,6e71d97dea2fd3a6,Senior Product Manager - Growth,Wallapop,"Barcelona, Barcelona provincia",NaN,NaN,híbrido,https://es.indeed.com/viewjob?jk=6e71d97dea2fd3a6,Wallapop is a Barcelona based scale-up driven ...,Indeed,"español, inglés, catalán",NaN,senior,grado,excel,"comunicación, liderazgo"


LINKEDIN

In [113]:
df_LinkedIn["modalidad"] = df_LinkedIn["ubicacion"].str.extract(r"\((.*?)\)")
df_LinkedIn["ubicacion"] = df_LinkedIn["ubicacion"].str.replace(r"\(.*?\)", "", regex=True).str.strip()
df_LinkedIn["idiomas"] = df_LinkedIn["descripcion"].apply(detectar_idiomas)
df_LinkedIn["experiencia_años"] = df_LinkedIn["descripcion"].apply(detectar_experiencia_años)
df_LinkedIn["experiencia_nivel"] = df_LinkedIn["descripcion"].apply(detectar_experiencia_nivel)
df_LinkedIn["nivel_educativo"] = df_LinkedIn["descripcion"].apply(detectar_nivel_educativo)
df_LinkedIn["hard_skills"] = df_LinkedIn["descripcion"].apply(detectar_hard_skills)
df_LinkedIn["soft_skills"] = df_LinkedIn["descripcion"].apply(detectar_soft_skills)
df_LinkedIn["tipo_contrato"] = df_LinkedIn["descripcion"].apply(detectar_tipo_contrato)
df_LinkedIn["descripcion"] = df_LinkedIn["descripcion"].str.strip("Acerca del empleo ")
df_LinkedIn


,titulo,empresa,ubicacion,url,id,fecha,descripcion,portal_web,modalidad,idiomas,experiencia_años,experiencia_nivel,nivel_educativo,hard_skills,soft_skills,tipo_contrato
0,📍 Data Scientist – ByRatings (Remote-Friendly ...,ByRatings,"Barcelona, Cataluña, España",https://www.linkedin.com/jobs/view/4368494721/,4368494721,NaN,\n¿Quiénes somos?\n\nEn ByRatings llevamos más...,LinkedIn,Híbrido,"español, inglés",10.0,becario,NaN,"python, pandas, numpy, scikit-learn",NaN,tiempo completo
1,Data Analyst Junior\nData Analyst Junior,Evolve,España,https://www.linkedin.com/jobs/view/4381466326/,4381466326,NaN,\nDescripción de la oferta\nEn nuestro Program...,LinkedIn,En remoto,NaN,NaN,becario,grado,"python, sql, power bi, pandas, scikit-learn",NaN,prácticas
2,Junior Data Analyst\nJunior Data Analyst with ...,Eurofragance SLU,"Sant Cugat del Vallès, Cataluña, España",https://www.linkedin.com/jobs/view/4380987091/,4380987091,NaN,\nEver dreamed of working where fragrances tel...,LinkedIn,Híbrido,NaN,1.0,becario,NaN,"python, sql, excel, power bi",NaN,prácticas
3,Data Analyst (Spanish) | $11/hr Remote\nData A...,Crossing Hurdles,España,https://www.linkedin.com/jobs/view/4376720386/,4376720386,NaN,\nPosition: LLM – AI Quality Analyst (Personal...,LinkedIn,En remoto,español,NaN,NaN,NaN,NaN,NaN,tiempo parcial
4,Data Analyst\nData Analyst with verification,SDG Group España,Barcelona y alrededores,https://www.linkedin.com/jobs/view/3417930563/,3417930563,NaN,\n¡Hola Data Lover! 💙\n\n¿Estás en último curs...,LinkedIn,Híbrido,inglés,NaN,NaN,NaN,"power bi, tableau","comunicación, pensamiento crítico",indefinido
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Data Scientist\nData Scientist,Sabadell Consumer Finance,"Sant Cugat del Vallès, Cataluña, España",https://www.linkedin.com/jobs/view/4374274153/,4374274153,NaN,\n¿Qué estamos buscando?\n\nActualmente nos en...,LinkedIn,Híbrido,NaN,4.0,NaN,master,"python, r, excel",trabajo en equipo,formación
96,Data Analyst - Internship - Barcelona\nData An...,papernest,"Barcelona, Cataluña, España",https://www.linkedin.com/jobs/view/4367834277/,4367834277,Hace 1 mes,"\nSince 2015, papernest has been transforming ...",LinkedIn,Híbrido,"español, inglés",NaN,becario,grado,"sql, tableau, docker, git",NaN,prácticas
97,Data Analyst\nData Analyst with verification,Adevinta,"Barcelona, Cataluña, España",https://www.linkedin.com/jobs/view/4378799550/,4378799550,NaN,\nJob Description\n\nWe are looking for an exc...,LinkedIn,Híbrido,NaN,3.0,NaN,NaN,"python, r, sql, tableau, pandas, numpy, scikit...",NaN,NaN
98,Research Analyst\nResearch Analyst,Alignerr,España,https://www.linkedin.com/jobs/view/4382365069/,4382365069,Hace 1 día,\nAbout The Role\n\nWe're seeking Research Ana...,LinkedIn,En remoto,NaN,NaN,NaN,NaN,NaN,comunicación,freelance


In [114]:
# Reordenar las columnas 
df_LinkedIn = df_LinkedIn.reindex(columns=["titulo", "empresa", "ubicacion", "modalidad", "tipo_contrato", "experiencia_años", "experiencia_nivel", "nivel_educativo", "hard_skills", "soft_skills", "idiomas","fecha","descripcion", "portal_web", "id", "url"])
df_Indeed = df_Indeed.reindex(columns= ["titulo", "empresa", "ubicacion", "modalidad", "tipo_contrato", "experiencia_años", "experiencia_nivel", "nivel_educativo", "hard_skills", "soft_skills", "idiomas", "salario", "descripcion", "portal_web", "id", "url"] )
df_Adzuna = df_Adzuna.reindex(columns = ["titulo", "empresa", "ubicacion", "modalidad", "tipo_contrato", "contract_type", "experiencia_años", "experiencia_nivel", "nivel_educativo", "hard_skills", "soft_skills", "idiomas", "fecha", "contrac_time", "category", "descripcion", "portal_web", "id", "url", "es_truncada"])


In [115]:
df_Adzuna.info()
df_Indeed.info()
df_LinkedIn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   titulo             76 non-null     object 
 1   empresa            73 non-null     object 
 2   ubicacion          76 non-null     object 
 3   modalidad          28 non-null     object 
 4   tipo_contrato      24 non-null     object 
 5   contract_type      4 non-null      object 
 6   experiencia_años   22 non-null     float64
 7   experiencia_nivel  58 non-null     object 
 8   nivel_educativo    20 non-null     object 
 9   hard_skills        43 non-null     object 
 10  soft_skills        43 non-null     object 
 11  idiomas            26 non-null     object 
 12  fecha              76 non-null     object 
 13  contrac_time       0 non-null      float64
 14  category           76 non-null     object 
 15  descripcion        76 non-null     object 
 16  portal_web         76 non-nu

In [116]:
# Funcion para normalizar los nulos
import numpy as np

def normalizar_nulos(df):
    df = df.replace([
        "", " ", "  ", "   ",
        "None", "none",
        "NaN", "nan",
        "N/A", "n/a",
        "null", "Null", "NULL",
        "-", "—", "Unknown"
    ], np.nan)
    return df


In [117]:
df_Adzuna = normalizar_nulos(df_Adzuna)
df_Indeed = normalizar_nulos(df_Indeed)
df_LinkedIn = normalizar_nulos(df_LinkedIn)

In [ ]:
# Guardar los dataframes _preproc preprocesados para continuar la depuración de datos
df_Adzuna.to_csv("df_Adzuna_preproc.csv", index=False)
df_Indeed.to_csv("df_Indeed_preproc.csv", index=False)
df_LinkedIn.to_csv("df_LinkedIn_preproc.csv", index=False)